# Imports

In [ ]:
import os
import sys
import time

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from uavloc.data.annotated import AnnotatedImage
from uavloc.datasets import dataset_ref
from uavloc.features.imagenet import normalise
from uavloc.features.segmentation import load_segmenter
from uavloc.plotting.segmentation import centre_window
from uavloc.segmentation.eval import (HOLDOUT_LOCATION, MEAN_KEY, STRIP_LOCATION, STRIP_ROWS, VAL_SETS,
                                      bilinear_argmax, evaluate, label_at_stride, load_val_sets, results_table,
                                      run_model)
from uavloc.segmentation.instances import building_table, merge_scores, merged_by_gap
from uavloc.segmentation.latency import model_latency, standard_tile
from uavloc.segmentation.losses import gap_band, gap_loss_terms
from uavloc.segmentation.monitor import crop_metrics, tiled_crops
from uavloc.segmentation.preview import log_segmenter_previews
from uavloc.track import REPO_ROOT, experiment_dir, log_history, log_run, set_seed, tensorboard_writer

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
EXP = "019-building-gap-loss"
NOTE = ""
LOG_RUN = True

GAP_WEIGHT = 0.0
GAP_RADIUS_PX = 10
STEPS = 400
LR = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
SAT_CROP_PX = 512
SAT_BATCH = 4
RARE_CROP_P = 0.3
GAP_PX = 64
RECALIBRATE_BN = True
BN_BATCHES = 60
BN_SEED = 123
SEED = 42
DEVICE = "mps"
EVAL_DEVICE = "cpu"
TRAIN_THREADS = 4
LOG_EVERY_STEPS = 25
MONITOR_TRAIN_CROPS = 32
MONITOR_SEED = 11
EVAL_BATCH = 8
PREVIEW_CROPS = 4
PREVIEW_PX = 256
PREVIEW_SEED = 7
DATASET = "ua-village-annotated"
INIT_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
GAP_BINS_M = [0, 1, 2, 3, 4, 6, 8, 12, 16]
LATENCY_TILE_DATE = "2024-06-06"
LATENCY_TILE_TOP_LEFT_PX = (1024, 1024)
LATENCY_BUDGET_MS = 125.0

In [ ]:
ORACLE, MODEL = "stride-8 oracle, bilinear", "LR-ASPP"
MERGE_KEYS = ("separable_merge_rate", "resolution_bound_merge_rate", "separate_share", "merged_share", "missed_share")

# Data

In [ ]:
set_seed(SEED)
torch.set_num_threads(TRAIN_THREADS)
DERIVED = REPO_ROOT / "data" / "derived"
ARTIFACTS = experiment_dir(EXP) / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

In [ ]:
sat = vs.load_satellite(DERIVED / DATASET, DERIVED)
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sat_train = [s for s in sat if not s.banned.all()]
sampler = vs.SatelliteSampler(sat_train, SAT_CROP_PX, RARE_CROP_P)
[(s.loc.split("-")[0], s.date) for s in sat_train]

In [ ]:
val_sets = load_val_sets(DATASET)
{name: len(images) for name, images in val_sets.items()}

In [ ]:
def centre_crops(images: list[AnnotatedImage], crop_px: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Normalised `(N, 3, P, P)` crops and `(N, P, P)` labels from the centre of each image."""
    rgb = [centre_window(a.rgb[None], crop_px)[0] for a in images]
    labels = [centre_window(a.label[None], crop_px)[0] for a in images]
    x = torch.from_numpy(np.stack([normalise(r).transpose(2, 0, 1) for r in rgb]))
    return x, torch.from_numpy(np.stack(labels).astype(np.int64))

In [ ]:
def round_robin(i: int) -> AnnotatedImage:
    """The i-th preview image: cycle over the val sets, then over each set's dates."""
    images = val_sets[VAL_SETS[i % len(VAL_SETS)]]
    return images[i // len(VAL_SETS) % len(images)]

In [ ]:
val_images = [round_robin(i) for i in range(PREVIEW_CROPS)]
previews = {"train": vs.batch_of(sampler, np.random.default_rng(PREVIEW_SEED), PREVIEW_CROPS),
            "val": centre_crops(val_images, SAT_CROP_PX)}
{split: tuple(x.shape) for split, (x, _) in previews.items()}

In [ ]:
monitor = {"train": vs.batch_of(sampler, np.random.default_rng(MONITOR_SEED), MONITOR_TRAIN_CROPS),
           "val": tiled_crops([a for name in VAL_SETS for a in val_sets[name]], SAT_CROP_PX)}
{split: tuple(x.shape) for split, (x, _) in monitor.items()}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / INIT_CHECKPOINT, DEVICE, architecture="lraspp")
n_params_initial = sum(p.numel() for p in model.parameters())
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=max(STEPS, 1), pct_start=WARMUP_FRAC)
n_params_initial

In [ ]:
def training_step(x: torch.Tensor, y: torch.Tensor) -> tuple[float, float]:
    """One optimiser step on gap-weighted CE + Dice; returns (loss, gap-band share of labelled pixels)."""
    band = gap_band(y, GAP_RADIUS_PX)
    logits = model(x.to(DEVICE))
    loss = gap_loss_terms(logits, y.to(DEVICE), band.to(DEVICE), GAP_WEIGHT)["loss"]
    opt.zero_grad()
    loss.backward()
    opt.step()
    sched.step()
    return loss.item(), band.float().mean().item()

In [ ]:
def loss_terms(logits: torch.Tensor, y: torch.Tensor) -> dict[str, torch.Tensor]:
    """The training loss and its parts on one batch, gap band computed from `y`."""
    return gap_loss_terms(logits, y, gap_band(y, GAP_RADIUS_PX).to(y.device), GAP_WEIGHT)

In [ ]:
def monitor_entry(step: int) -> dict[str, float]:
    """Every loss term, mIoU and building IoU on the fixed train and val monitor crops."""
    return {"step": step, **{k: v for split, crops in monitor.items()
                             for k, v in crop_metrics(model, crops, loss_terms, EVAL_BATCH, DEVICE, split).items()}}

# Run

In [ ]:
writer = tensorboard_writer(EXP)
rng = np.random.default_rng(SEED)
model.train()
history, window, t0 = [monitor_entry(0)], [], time.time()
log_history(writer, history[-1])
log_segmenter_previews(writer, model, previews, 0, DEVICE)
progress = tqdm(range(1, STEPS + 1), desc="train")
for step in progress:
    lr = sched.get_last_lr()[0]
    loss, share = training_step(*vs.batch_of(sampler, rng, SAT_BATCH))
    window.append(share)
    log_history(writer, {"step": step, "train_step_loss": loss, "train_lr": lr})
    progress.set_postfix(loss=f"{loss:.3f}", val_loss=f"{history[-1]['val_loss']:.3f}")
    if step % LOG_EVERY_STEPS == 0 or step == STEPS:
        history.append({**monitor_entry(step), "train_band_share": float(np.mean(window)),
                        "minutes": (time.time() - t0) / 60})
        log_history(writer, history[-1])
        log_segmenter_previews(writer, model, previews, step, DEVICE)
        window = []
train_minutes = (time.time() - t0) / 60

In [ ]:
if RECALIBRATE_BN and STEPS:
    model = vs.domain_bn(model, sampler, SAT_BATCH, BN_BATCHES, BN_SEED, torch.device(DEVICE))
model.eval()
log_segmenter_previews(writer, model, previews, STEPS, DEVICE, "segmentation_final")
checkpoint = ARTIFACTS / f"model-gap{GAP_WEIGHT:g}-seed{SEED}-steps{STEPS}.pt" if STEPS else None
if checkpoint:
    torch.save(model.state_dict(), checkpoint)
checkpoint

In [ ]:
cases = run_model(model.to(EVAL_DEVICE), val_sets, EVAL_DEVICE)
results = evaluate(bilinear_argmax, cases)
results_table(results).round(3)

In [ ]:
def case_table(name: str, case) -> pd.DataFrame:
    """Per labelled building of one val case: gap and status under the stride-8 oracle and the model."""
    label = case.image.label
    maps = {ORACLE: label_at_stride(label, "bilinear"), MODEL: bilinear_argmax(case.output, case.image.rgb)}
    return building_table(label, maps).assign(set=name, date=case.image.date)

In [ ]:
buildings = pd.concat([case_table(name, c) for name, items in cases.items() for c in items], ignore_index=True)
merges = {name: merge_scores(frame, MODEL, ORACLE) for name, frame in buildings.groupby("set")}
merges[MEAN_KEY] = {k: float(np.mean([merges[n][k] for n in VAL_SETS])) for k in MERGE_KEYS}
pd.DataFrame(merges).T.round(3)

In [ ]:
by_gap = merged_by_gap(buildings, [ORACLE, MODEL], GAP_BINS_M)
by_gap.round(3)

In [ ]:
tile_image = next(a for a in val_sets["bushcha"] if a.date == LATENCY_TILE_DATE)
tile = standard_tile(tile_image.rgb, *LATENCY_TILE_TOP_LEFT_PX)
latency = model_latency(model, tile)
assert sum(p.numel() for p in model.parameters()) == n_params_initial
latency

# Plots

In [ ]:
fig_loss, axes = plt.subplots(1, 2, figsize=(11, 3.5))
frame = pd.DataFrame(history)
for split in ("train", "val"):
    axes[0].plot(frame["step"], frame[f"{split}_loss"], "-o", ms=3, label=f"{split} loss")
    axes[1].plot(frame["step"], frame[f"{split}_miou"], "-o", ms=3, label=f"{split} mIoU")
axes[0].set_title(f"gap-weighted CE + Dice, w0 = {GAP_WEIGHT:g}, seed {SEED}")
axes[1].set_title("mIoU on monitor crops")
for ax in axes:
    ax.set_xlabel("step"); ax.grid(alpha=.25); ax.legend()
fig_loss.tight_layout()

In [ ]:
fig_gap, ax = plt.subplots(figsize=(7, 4))
centres = [(a + b) / 2 for a, b in zip(GAP_BINS_M, GAP_BINS_M[1:] + [GAP_BINS_M[-1] + 4])]
for key in (ORACLE, MODEL):
    ax.plot(centres, by_gap[key].values, "-o", label=key)
ax.set_xlabel("gap to nearest labelled building (m)"); ax.set_ylabel("share merged")
ax.set_title(f"merged share by gap, w0 = {GAP_WEIGHT:g}, seed {SEED}"); ax.grid(alpha=.3); ax.legend(fontsize=8)
fig_gap.tight_layout()

# Log

In [ ]:
mean = results[MEAN_KEY]
metrics = {
    "separable_merge_rate_mean": merges[MEAN_KEY]["separable_merge_rate"],
    "miou_mean": mean["miou"],
    "building_missed_share_mean": merges[MEAN_KEY]["missed_share"],
    "building_iou_mean": float(np.mean([results[n]["iou_per_class"]["building"] for n in VAL_SETS])),
    "building_merged_share_mean": merges[MEAN_KEY]["merged_share"],
    "building_separate_share_mean": merges[MEAN_KEY]["separate_share"],
    "resolution_bound_merge_rate_mean": merges[MEAN_KEY]["resolution_bound_merge_rate"],
    "boundary_iou_mean": mean["biou"], "bf1_1px_mean": mean["bf1_1px"], "bf1_2px_mean": mean["bf1_2px"],
    "mdice_mean": mean["mdice"],
    "model_cpu_ms": latency.median_ms, "within_latency_budget": latency.median_ms <= LATENCY_BUDGET_MS,
    "train_minutes": train_minutes,
    "gap_band_share_train": float(np.mean([h["train_band_share"] for h in history[1:]])) if STEPS else None,
    **{f"separable_merge_rate_{n}": merges[n]["separable_merge_rate"] for n in VAL_SETS},
    **{f"building_iou_{n}": results[n]["iou_per_class"]["building"] for n in VAL_SETS},
    **{f"miou_{n}": results[n]["miou"] for n in VAL_SETS},
    "merge_scores": {n: merges[n] for n in VAL_SETS},
    "merged_by_gap": {str(k): v for k, v in by_gap.to_dict(orient="index").items()},
    "iou_per_class": {n: results[n]["iou_per_class"] for n in VAL_SETS},
    "biou_per_class": {n: results[n]["biou_per_class"] for n in VAL_SETS},
    "loss_history": history,
}
{k: metrics[k] for k in list(metrics)[:4]}

In [ ]:
RUN_CONFIG = {"dataset": dataset_ref(DATASET), "init_checkpoint": INIT_CHECKPOINT, "gap_weight": GAP_WEIGHT,
              "gap_radius_px": GAP_RADIUS_PX, "steps": STEPS, "lr": LR, "weight_decay": WEIGHT_DECAY,
              "warmup_frac": WARMUP_FRAC, "crop_px": SAT_CROP_PX, "batch": SAT_BATCH, "rare_crop_p": RARE_CROP_P,
              "gap_px": GAP_PX, "recalibrate_bn": RECALIBRATE_BN, "bn_batches": BN_BATCHES, "bn_seed": BN_SEED,
              "train_threads": TRAIN_THREADS, "eval_device": EVAL_DEVICE}
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_loss, fig_gap], seed=SEED,
              checkpoint=str(checkpoint.relative_to(REPO_ROOT)) if checkpoint else None,
              device=DEVICE, writer=writer) if LOG_RUN else {"run": "dry"}
writer.close()
run["run"]